In [ ]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

In [ ]:
from corr_vars_widget import ObsWidget
import polars as pl

from datetime import datetime

df = pl.DataFrame({"a": [1, 2, 3] * 10_000, "b": ["This", "is a", "test"] * 10_000})
creation_time = datetime.now()
obs = ObsWidget(df, "ICU Stay", creation_time=creation_time)
obs

In [ ]:
print(obs)

In [ ]:
from corr_vars_widget import ObsmWidget
import polars as pl

df = pl.DataFrame({"a": [1, 2, 3] * 10_000, "b": ["This", "is a", "test"] * 10_000})
obsm = ObsmWidget({"df1": df, "df2": df.with_columns(pl.col("a") * 2)})
obsm

In [ ]:
print(obsm)

In [ ]:
print(obsm._tables)

In [ ]:
obsm.sql

In [ ]:
obsm.data

In [ ]:
from corr_vars_widget import JsonWidget

data = {
    "a": [1, 2, 3],
    "b": ["This", "is a", "test"],
    "bool": True,
    "null": None,
    "nested": {
        "c": [4, 5, 6],
        "d": {"e": "Nested value"},
        "empty_list": [],
        "empty_dict": {},
    },
}
json_widget = JsonWidget(data)
json_widget

In [ ]:
var_configs = {
    "icu_length_of_stay": {
        "type": "derived_static",
        "compatible_with": ["icu_stay"],
        "requires": ["icu_admission", "icu_discharge"],
        "expression": "IF(icu_discharge - icu_admission IS NULL, NULL, GREATEST((EXTRACT(EPOCH FROM icu_discharge) - EXTRACT(EPOCH FROM icu_admission)) / (24 * 60 * 60), 1.0))",
    },
    "hospital_length_of_stay": {
        "type": "derived_static",
        "requires": ["hospital_admission", "hospital_discharge"],
        "expression": "IF(hospital_discharge - hospital_admission IS NULL, NULL, GREATEST((EXTRACT(EPOCH FROM hospital_discharge) - EXTRACT(EPOCH FROM hospital_admission)) / (24 * 60 * 60), 1.0))",
    },
    "campus_id": {
        "type": "derived_static",
        "compatible_with": ["icu_stay"],
        "requires": ["icu_id"],
        "expression": "SUBSTR(ARRAY_GET(icu_id, 1), 1, 1)",
    },
    "blood_sodium": {
        "type": "native_dynamic",
        "table": "it_ishmed_labor",
        "where": "(c_katalog_leistungtext IN ('Natrium HP', 'BGA-Natrium', 'Natrium (POC)', 'NATRIUM')) OR (c_katalog_leistungtext LIKE 'NATRIUM(BG)%') OR (c_katalog_leistungtext='Natrium' AND c_leistungnr IN ('N_101000', 'N_303011', 'C_NA', 'N_204100', 'N_701000'))",
        "value_dtype": "DOUBLE",
        "cleaning": {"value": {"low": 80, "high": 190}},
    },
    "blood_hematocrit": {
        "type": "native_dynamic",
        "table": "it_ishmed_labor",
        "where": "c_katalog_leistungtext LIKE '%Hämatokr%' AND NOT (c_leistungnr = 'H_HKTI' OR c_leistungnr = 'N_204140')",
        "value_dtype": "FLOAT",
        "cleaning": {"value": {"low": 0.0, "high": 1.0}},
    },
}

In [ ]:
json_widget = JsonWidget(var_configs)
json_widget